# 🔧 Data Pipeline Design — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A data pipeline is a factory assembly line. Raw materials (events, records, files) enter one end. Each station on the line does one job: clean, enrich, validate, aggregate. Finished product exits at the other end into a data store. The key design decisions are: do you run the line on a fixed schedule (batch), or do you process every part the moment it arrives (streaming)? The Lambda architecture runs two parallel lines — one slow/accurate, one fast/approximate — and merges the outputs. Kappa says: just run one really good streaming line and replay history when you need to reprocess.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Data Pipeline Design? The Visual Model](#1) |
| 2 | [Creating / Setup — Pipeline Building Blocks](#2) |
| 3 | [The Core API — Source, Transform, Sink](#3) |
| 4 | [Decision Map — Batch vs Streaming vs Hybrid](#4) |
| 5 | [Pattern 1: Batch ETL Pipeline](#5) |
| 6 | [Pattern 2: Streaming Pipeline](#6) |
| 7 | [Pattern 3: Lambda Architecture](#7) |
| 8 | [Pattern 4: Kappa Architecture](#8) |
| 9 | [Pattern 5: Pipeline Observability & SLAs](#9) |
| 10 | [The Data Pipeline Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. 🗺️ What Is Data Pipeline Design? The Visual Model

```
BATCH PIPELINE (scheduled, high throughput):

  [Source DB / S3]  ──(extract)──►  [Transform/Spark]  ──(load)──►  [DW / S3]
  Runs every: 1hr, 1day                                              Latency: minutes–hours
  Pros: simple, efficient, idempotent                                Cons: stale data

STREAMING PIPELINE (event-driven, low latency):

  [Kafka Topic]  ──(consume)──►  [Flink/Spark SS]  ──(produce)──►  [Kafka/DB]
  Triggers on: every event                                           Latency: ms–seconds
  Pros: fresh data, real-time decisions                              Cons: complex, stateful

LAMBDA ARCHITECTURE (hybrid):

  [Source] ──────────────────────────────────────────────►
               │                         │
        [Batch Layer]              [Speed Layer]
        (Spark, HDFS)              (Flink, Kafka)
        complete/accurate           fast/approximate
               │                         │
               └──────────►  [Serving Layer]  ◄───────────┘
                            (merges batch + speed views)

KAPPA ARCHITECTURE (streaming-only):

  [Kafka log (long retention)]  ──►  [Streaming Processor]  ──►  [Serving]
  Reprocess: replay Kafka from offset 0 into new consumer group
  Advantage: one system to maintain instead of two

KEY TRADEOFFS:
  Latency  ◄──────────────────────────────────► Throughput
  Batch=high throughput  Streaming=low latency  Micro-batch=compromise
```

<a id='2'></a>
## 2. 🔧 Creating / Setup — Pipeline Building Blocks

In [ ]:
# PIPELINE BUILDING BLOCKS
# Every pipeline shares three conceptual layers: Source, Transform, Sink

from dataclasses import dataclass, field
from typing import List, Callable, Any, Optional
from datetime import datetime
import time

@dataclass
class PipelineRecord:
    id: str
    timestamp: str
    payload: dict
    source: str = "unknown"
    error: Optional[str] = None   # set if transformation failed

@dataclass
class PipelineMetrics:
    processed: int = 0
    failed: int = 0
    skipped: int = 0
    start_time: float = field(default_factory=time.time)

    def throughput(self):
        elapsed = time.time() - self.start_time
        return self.processed / elapsed if elapsed > 0 else 0

    def error_rate(self):
        total = self.processed + self.failed
        return self.failed / total if total > 0 else 0

class PipelineStage:
    def __init__(self, name: str, transform_fn: Callable):
        self.name = name
        self.transform = transform_fn   # function: record → record (or None to drop)
        self.metrics = PipelineMetrics()

    def process(self, record: PipelineRecord) -> Optional[PipelineRecord]:
        try:
            result = self.transform(record)
            self.metrics.processed += 1
            return result
        except Exception as e:
            record.error = str(e)
            self.metrics.failed += 1
            return None           # failed records go to dead letter queue

# Demo: create a simple 3-stage pipeline
def validate_stage(record):
    if not record.payload.get("user_id"):
        raise ValueError("missing user_id")
    return record

def enrich_stage(record):
    record.payload["processed_at"] = datetime.now().isoformat()
    return record

def normalize_stage(record):
    record.payload["user_id"] = str(record.payload["user_id"]).lower()
    return record

stages = [
    PipelineStage("validate", validate_stage),
    PipelineStage("enrich", enrich_stage),
    PipelineStage("normalize", normalize_stage),
]

print(f"Pipeline stages: {[s.name for s in stages]}")
print("Building blocks: PipelineRecord, PipelineMetrics, PipelineStage defined.")

<a id='3'></a>
## 3. ⚡ The Core API — Source, Transform, Sink

```
CONCEPT                  BATCH EXAMPLE           STREAMING EXAMPLE
─────────────────────────────────────────────────────────────────────────
Source                   S3 file / JDBC query    Kafka consumer / Kinesis
Transform                Spark DataFrame ops     Flink map/filter/window
Sink                     S3 / Redshift / DW      Kafka topic / DynamoDB
Trigger                  Cron schedule           Event arrival
State                    Stateless (usually)     Stateful (aggregations)
Fault tolerance          Idempotent re-runs      Checkpointing + offsets
Latency                  Minutes to hours        Milliseconds to seconds
─────────────────────────────────────────────────────────────────────────

IDEMPOTENCY RULE (critical for batch):
  Running the pipeline twice on the same input → same output, no duplicates.
  Achieve with: INSERT OVERWRITE (not APPEND), use unique keys, MERGE/UPSERT.

EXACTLY-ONCE DELIVERY OPTIONS:
  At-most-once:  drop on failure (fire and forget) — use for metrics/logging
  At-least-once: retry on failure, consumer deduplicates — most common
  Exactly-once:  Kafka transactions + idempotent producers — expensive

CHECKPOINTING (streaming):
  Persist current state + consumer offset to durable storage periodically.
  On failure: restore from checkpoint, replay from saved offset.
  Frequency: trade-off between recovery time and overhead.

THINGS YOU DO NOT DO:
❌  Append-only sinks without dedup → duplicate records on retry
✅  UPSERT / INSERT OVERWRITE / partition-level overwrite for idempotency
❌  Process entire historical dataset on every run
✅  Watermark or high-water-mark: only process records since last run
❌  One giant monolithic transform function → hard to debug, no partial progress
✅  Chain of small, testable stages with metrics at each step
```

In [ ]:
# LIVE DEMO: idempotency patterns for batch pipelines

# Pattern 1: High-water mark — only process new records since last run
class HighWaterMark:
    def __init__(self, initial=None):
        self.last_processed_at = initial  # timestamp of last successful run

    def get_query(self, table):
        if self.last_processed_at is None:
            return f"SELECT * FROM {table}"  # first run — full load
        return (f"SELECT * FROM {table} "
                f"WHERE updated_at > '{self.last_processed_at}'")

    def advance(self, new_mark):
        self.last_processed_at = new_mark   # only advance AFTER successful load

hwm = HighWaterMark("2026-03-01")
print("High-water mark query:", hwm.get_query("orders"))
hwm.advance("2026-03-20")
print("After advance:", hwm.get_query("orders"))

print()

# Pattern 2: Partition-level overwrite for idempotency
class PartitionWriter:
    def write(self, data, partition_date):
        # overwrite entire partition — idempotent by design
        # if we re-run with same partition_date, old data is replaced, not appended
        path = f"s3://bucket/events/date={partition_date}/"
        print(f"OVERWRITE {path}  ({len(data)} records)")
        print(f"  → safe to retry: same data will produce same partition")

pw = PartitionWriter()
pw.write(["rec1", "rec2", "rec3"], "2026-03-20")

print()

# Pattern 3: Dead letter queue for failed records
class PipelineWithDLQ:
    def __init__(self):
        self.dlq = []              # dead letter queue — failed records for inspection
        self.processed = []        # successfully processed records

    def process(self, records, transform_fn):
        for r in records:
            try:
                self.processed.append(transform_fn(r))
            except Exception as e:
                self.dlq.append({"record": r, "error": str(e)})  # don't drop — inspect
        print(f"Processed: {len(self.processed)}, DLQ: {len(self.dlq)}")

def bad_transform(r):
    if r == "bad":
        raise ValueError("invalid record")
    return r.upper()

pipeline = PipelineWithDLQ()
pipeline.process(["ok", "bad", "good"], bad_transform)
print("DLQ contents:", pipeline.dlq)

<a id='4'></a>
## 4. 🗂️ Decision Map — Batch vs Streaming vs Hybrid

```
REQUIREMENT                           ARCHITECTURE CHOICE
──────────────────────────────────────────────────────────────────────
Latency tolerance > 15 min            Batch ETL (simplest, cheapest)
Latency tolerance 1–15 min            Micro-batch (Spark Structured Streaming)
Latency tolerance < 1 min             True streaming (Flink / Kafka Streams)
Need both historical + real-time      Lambda Architecture
Want one system, tolerate replay      Kappa Architecture
ML feature store with fresh features  Streaming + feature serving layer
──────────────────────────────────────────────────────────────────────

TRIGGER SELECTION:
  Cron-based      → batch, predictable load, easy scheduling
  Event-driven    → streaming, immediate reaction, variable load
  File arrival    → S3 event triggers Lambda/Glue job
  SLA-driven      → set trigger frequency = SLA / 2 (safety margin)

FAULT TOLERANCE STRATEGY:
  Batch:     idempotent re-runs, partition overwrite, high-water mark
  Streaming: offset commits (at-least-once) or Kafka transactions (exactly-once)
  Both:      dead letter queue for poison records, alerting on DLQ depth

INTERVIEW TRAP: "We need real-time analytics."
  Challenge the requirement: near-real-time (5 min micro-batch) is 10x simpler
  than true real-time streaming. Ask: what decision does the data drive?
  If a human dashboard, 5 min is fine. If fraud detection, you need sub-second.
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Batch ETL Pipeline

---

```
SCENARIO:  Nightly pipeline processes all orders from the past 24 hours,
           enriches with product catalog, aggregates by region, loads to DW.

DESIGN DECISIONS:
  Source:       production DB, JDBC read with push-down predicates
  Extract:      partition by date, only load today's partition
  Transform:    join with product catalog (broadcast join if catalog is small)
  Load:         overwrite today's DW partition (idempotent)
  Trigger:      cron at 01:00 UTC daily
  SLA:          data available by 06:00 UTC

STEP-BY-STEP EXECUTION:
  1. Read high-water mark from metadata table: last_run = "2026-03-19"
  2. Extract: SELECT * FROM orders WHERE order_date = '2026-03-20'
  3. Validate: drop records with null order_id, log count to metrics
  4. Enrich: LEFT JOIN product_catalog ON orders.product_id
  5. Aggregate: GROUP BY region, date → sum(revenue), count(orders)
  6. Load: INSERT OVERWRITE s3://dw/orders/date=2026-03-20/ (overwrite)
  7. Update high-water mark to '2026-03-20'
  8. Send success notification to monitoring

IDEMPOTENCY KEY: step 6 uses INSERT OVERWRITE, not APPEND.
  Re-running on the same date produces the same output — safe for retries.

FAILURE HANDLING:
  - Steps 1-6 fail → retry from step 1 (high-water mark unchanged)
  - Step 7 fails → pipeline succeeds but mark not updated → will re-run tomorrow
  - Guard: idempotent load means double-run on same partition = no duplicates
```

In [ ]:
# BATCH ETL PIPELINE SIMULATION

from dataclasses import dataclass, field
from typing import List, Dict, Optional
import datetime

@dataclass
class BatchJob:
    name: str
    schedule: str                       # cron expression e.g. "0 1 * * *"
    sla_hours: int                      # must complete within N hours
    last_watermark: Optional[str] = None
    run_log: List[Dict] = field(default_factory=list)

    def run(self, partition_date: str, records: List[Dict]) -> bool:
        run_id = f"{self.name}_{partition_date}"
        print(f"[START] {run_id}")

        # STEP 1: Extract — filter by partition date (incremental)
        filtered = [r for r in records if r.get("date") == partition_date]
        print(f"  [EXTRACT] {len(filtered)} records for date={partition_date}")

        # STEP 2: Validate — drop records missing required fields
        valid = [r for r in filtered if r.get("order_id") and r.get("amount")]
        dropped = len(filtered) - len(valid)
        if dropped > 0:
            print(f"  [VALIDATE] dropped {dropped} invalid records → DLQ")

        # STEP 3: Transform — aggregate by region
        agg = {}
        for r in valid:
            region = r.get("region", "unknown")
            agg[region] = agg.get(region, 0) + r["amount"]
        print(f"  [TRANSFORM] aggregated into {len(agg)} regions: {agg}")

        # STEP 4: Load — partition overwrite (idempotent)
        partition_path = f"s3://dw/orders/date={partition_date}/"
        print(f"  [LOAD] OVERWRITE {partition_path}  ({len(agg)} rows)")

        # STEP 5: Advance watermark ONLY after successful load
        self.last_watermark = partition_date
        self.run_log.append({"run_id": run_id, "status": "SUCCESS", "rows": len(valid)})
        print(f"[DONE] watermark advanced to {partition_date}")
        return True

# Demo: run the nightly orders pipeline
orders_pipeline = BatchJob(name="orders_etl", schedule="0 1 * * *", sla_hours=5)

sample_records = [
    {"order_id": "o1", "date": "2026-03-20", "region": "west", "amount": 100},
    {"order_id": "o2", "date": "2026-03-20", "region": "east", "amount": 200},
    {"order_id": None, "date": "2026-03-20", "region": "west", "amount": 50},  # invalid
    {"order_id": "o3", "date": "2026-03-20", "region": "west", "amount": 150},
    {"order_id": "o4", "date": "2026-03-19", "region": "east", "amount": 99},  # wrong date
]

orders_pipeline.run("2026-03-20", sample_records)
print()
print("Re-run (simulating retry) — should produce same output (idempotent):")
orders_pipeline.run("2026-03-20", sample_records)  # same result, no duplicates

<a id='6'></a>
## 6. 🧩 Pattern 2: Streaming Pipeline

---

```
SCENARIO:  Real-time fraud detection: every payment event must be scored
           within 500ms and flagged if score > threshold.

DESIGN DECISIONS:
  Source:       Kafka topic "payments" (partitioned by user_id)
  Processing:   Flink job — stateless per-event scoring
  Enrichment:   Redis lookup for user profile (P99 < 5ms)
  Output:       Kafka topic "flagged_payments" + DynamoDB for audit
  Parallelism:  Match Kafka partition count (scale consumer to num_partitions)
  Checkpointing: every 30s to S3 — recovery point if Flink restarts

EVENT FLOW:
  payment arrives → consumer reads from partition
  → deserialize JSON
  → enrich: lookup user risk score from Redis
  → compute: rule engine + ML model score
  → route: score > 0.8 → "flagged_payments" topic
             score ≤ 0.8 → "approved_payments" topic
  → commit offset (at-least-once delivery)
  Total latency target: < 500ms P99

BACKPRESSURE:
  If consumer falls behind (lag grows), Kafka retains messages.
  Horizontal scale: add more consumer instances up to partition count.
  Vertical: increase parallelism of Flink operators.
  Alert: consumer_lag > 10000 events → PagerDuty

OFFSET MANAGEMENT:
  Commit AFTER processing (at-least-once): may reprocess on restart, must dedup.
  Commit BEFORE processing (at-most-once): may lose events on crash.
  Exactly-once: Kafka transactions (expensive, use only for financial data).
```

In [ ]:
# STREAMING PIPELINE SIMULATION
# Simulates the event-by-event processing model

from collections import deque
from dataclasses import dataclass, field
from typing import Callable, List, Dict, Optional

@dataclass
class KafkaMessage:
    topic: str
    partition: int
    offset: int
    key: str
    value: Dict

class StreamProcessor:
    def __init__(self, consumer_group: str, checkpoint_interval: int = 100):
        self.consumer_group = consumer_group
        self.checkpoint_interval = checkpoint_interval
        self.committed_offsets: Dict[int, int] = {}   # partition → last committed offset
        self.processed_count = 0
        self.output_topics: Dict[str, List] = {}      # topic → messages sent

    def add_output_topic(self, topic_name):
        self.output_topics[topic_name] = []

    def process_stream(self, messages: List[KafkaMessage], process_fn: Callable):
        for msg in messages:
            # PROCESS: apply business logic
            result = process_fn(msg.value)
            self.processed_count += 1

            # ROUTE: send to appropriate output topic
            if result:
                topic, output_record = result
                if topic in self.output_topics:
                    self.output_topics[topic].append(output_record)

            # COMMIT OFFSET after processing (at-least-once semantics)
            self.committed_offsets[msg.partition] = msg.offset

            # CHECKPOINT: persist state periodically
            if self.processed_count % self.checkpoint_interval == 0:
                self._checkpoint()

    def _checkpoint(self):
        # in production: write to S3/ZooKeeper for fault tolerance
        print(f"  [CHECKPOINT] offsets={self.committed_offsets} processed={self.processed_count}")

    def consumer_lag(self, latest_offsets: Dict[int, int]) -> int:
        # how far behind are we from the latest messages?
        lag = 0
        for partition, latest in latest_offsets.items():
            committed = self.committed_offsets.get(partition, -1)
            lag += max(0, latest - committed - 1)
        return lag

# Demo: fraud detection streaming pipeline
def fraud_detector(payment: Dict):
    # simple rule: flag if amount > 5000 or unusual country
    score = 0.0
    if payment.get("amount", 0) > 5000:
        score += 0.6
    if payment.get("country") not in ["US", "CA", "GB"]:
        score += 0.3
    topic = "flagged_payments" if score > 0.8 else "approved_payments"
    return topic, {**payment, "fraud_score": round(score, 2)}

processor = StreamProcessor("fraud-detection-group", checkpoint_interval=3)
processor.add_output_topic("flagged_payments")
processor.add_output_topic("approved_payments")

events = [
    KafkaMessage("payments", 0, 0, "u1", {"user": "u1", "amount": 100, "country": "US"}),
    KafkaMessage("payments", 0, 1, "u2", {"user": "u2", "amount": 9999, "country": "XX"}),
    KafkaMessage("payments", 0, 2, "u3", {"user": "u3", "amount": 250, "country": "CA"}),
    KafkaMessage("payments", 0, 3, "u4", {"user": "u4", "amount": 6000, "country": "US"}),
]

processor.process_stream(events, fraud_detector)
print(f"\nFlagged: {processor.output_topics['flagged_payments']}")
print(f"Approved: {processor.output_topics['approved_payments']}")
print(f"Consumer lag: {processor.consumer_lag({0: 10})} events behind")

<a id='7'></a>
## 7. 🧩 Pattern 3: Lambda Architecture

---

```
SCENARIO:  E-commerce: need both real-time product recommendations (speed layer)
           and accurate monthly revenue reports (batch layer).

LAYER RESPONSIBILITIES:
  Batch Layer:   processes ALL historical data periodically
                 high accuracy, high latency (hours), stores batch views
                 tool: Spark on EMR, results to S3 / Redshift

  Speed Layer:   processes only RECENT data (since last batch)
                 low accuracy (may have late arrivals), low latency (seconds)
                 tool: Flink or Spark Structured Streaming, results to Redis

  Serving Layer: merges batch views + speed views at query time
                 returns: batch_result UNION speed_result (with dedup)
                 tool: Druid, Pinot, or custom API layer

DATA FLOW DIAGRAM:
  Events ──►  [Kafka log]  ──────────────────────────────────────────►
                  │                              │
         [Batch Layer: Spark]          [Speed Layer: Flink]
         reads ALL from S3             reads LAST 1hr from Kafka
         runs every 6 hours            runs continuously
                  │                              │
         [Batch Views: S3]             [Speed Views: Redis]
                  └──────────►  [Serving Layer]  ◄───────────────────┘
                               QUERY = batch_view + speed_view (merged)

ADVANTAGE: batch layer provides accurate historical view;
           speed layer fills in the gap since last batch run.
DISADVANTAGE: maintaining two separate codebases for same logic is expensive.
              Lambda's biggest criticism: duplicated business logic.
```

In [ ]:
# LAMBDA ARCHITECTURE SIMULATION

from dataclasses import dataclass, field
from typing import List, Dict
import datetime

@dataclass
class BatchView:
    """Output of the batch layer — covers all data up to last batch run."""
    computed_at: str               # when the batch job finished
    covers_through: str            # data through this timestamp
    data: Dict[str, float]         # aggregated results: {key: value}

@dataclass
class SpeedView:
    """Output of the speed layer — covers only data AFTER last batch run."""
    data: Dict[str, float]         # partial, may have late arrivals missing
    since: str                     # covers events after this timestamp

class LambdaServingLayer:
    def __init__(self):
        self.batch_view: BatchView = None
        self.speed_view: SpeedView = None

    def update_batch(self, batch: BatchView):
        self.batch_view = batch
        print(f"[BATCH UPDATE] covers through {batch.covers_through}")

    def update_speed(self, speed: SpeedView):
        self.speed_view = speed
        print(f"[SPEED UPDATE] {len(speed.data)} keys updated")

    def query(self, metric_key: str) -> Dict:
        """Merge batch + speed views at query time."""
        batch_val = self.batch_view.data.get(metric_key, 0) if self.batch_view else 0
        speed_val = self.speed_view.data.get(metric_key, 0) if self.speed_view else 0
        # merge: batch covers historical, speed covers recent gap
        merged = batch_val + speed_val
        return {
            "key": metric_key,
            "batch_value": batch_val,
            "speed_value": speed_val,
            "merged": merged,
            "note": "batch=historical, speed=recent gap"
        }

# Demo: revenue dashboard using Lambda architecture
serving = LambdaServingLayer()

# Batch layer runs at 06:00, covers all orders through midnight
batch = BatchView(
    computed_at="2026-03-20T06:00:00",
    covers_through="2026-03-20T00:00:00",
    data={"revenue_west": 125000.0, "revenue_east": 98000.0}
)
serving.update_batch(batch)

# Speed layer has processed events from midnight to now (morning traffic)
speed = SpeedView(
    since="2026-03-20T00:00:00",
    data={"revenue_west": 18500.0, "revenue_east": 12300.0}
)
serving.update_speed(speed)

# Query at 08:00 — gets complete picture
print("\nQuery result (08:00 AM):")
result = serving.query("revenue_west")
for k, v in result.items():
    print(f"  {k}: {v}")

print(f"\nTotal west revenue: ${result['merged']:,.0f}")
print("(batch=last night's data, speed=today's morning data merged at query time)")

<a id='8'></a>
## 8. 🧩 Pattern 4: Kappa Architecture

---

```
SCENARIO:  Same e-commerce use case, but team wants one system to maintain.
           History reprocessing happens by replaying the Kafka log.

CORE INSIGHT:
  Kafka retains ALL events with configurable retention (days, weeks, forever).
  Reprocessing = start a new consumer group at offset 0, replay everything.
  Lambda's batch layer becomes: "replay streaming job from beginning of log."

DATA FLOW:
  Events ──►  [Kafka log (long retention: e.g., 30 days)]  ──────────────────►
                   ▲                         │
                   │ replay from offset 0    ▼
              [reprocessing]         [Streaming Processor v1]
                                     (current production)

  To deploy a new version:
    1. Start Streaming Processor v2 with new logic on SAME Kafka topic
    2. v2 reads from offset 0 (catches up to present)
    3. Once v2 catches up, cut traffic over from v1 → v2
    4. Decommission v1

REPROCESSING PATTERN:
  new_consumer_group = "processor_v2_reprocess"
  start_offset = 0  # beginning of log (or any point in history)
  → produces new output to new sink or with version tag
  → validate output before switching production traffic

ADVANTAGE over Lambda:
  - One codebase: streaming logic only (no duplicate batch + speed logic)
  - Simpler ops: one Flink/Spark cluster instead of two

DISADVANTAGE:
  - Kafka retention cost (storing weeks of raw events is expensive)
  - Replay can be slow for large historical datasets
  - Not ideal if historical data is not in Kafka (e.g., pre-dates event stream)
```

In [ ]:
# KAPPA ARCHITECTURE SIMULATION

from dataclasses import dataclass, field
from typing import List, Dict, Optional

@dataclass
class KafkaTopic:
    name: str
    retention_days: int
    log: List[Dict] = field(default_factory=list)   # the immutable event log

    def append(self, event: Dict):
        event["offset"] = len(self.log)  # assign monotonic offset
        self.log.append(event)

    def read_from(self, consumer_group: str, start_offset: int = 0):
        # returns all events from start_offset to end
        return self.log[start_offset:]

class KappaProcessor:
    def __init__(self, version: str, process_fn):
        self.version = version
        self.process_fn = process_fn    # business logic — the only code that changes
        self.output: Dict[str, float] = {}

    def run(self, topic: KafkaTopic, start_offset: int = 0):
        events = topic.read_from(f"processor_{self.version}", start_offset)
        print(f"[{self.version}] processing {len(events)} events from offset {start_offset}")
        for event in events:
            self.process_fn(event, self.output)
        print(f"[{self.version}] output: {self.output}")
        return self.output

# Simulate the Kafka event log (long retention)
payments_topic = KafkaTopic("payments", retention_days=30)
for pay in [
    {"user": "u1", "region": "west", "amount": 100},
    {"user": "u2", "region": "east", "amount": 200},
    {"user": "u3", "region": "west", "amount": 150},
    {"user": "u4", "region": "east", "amount": 50},
]:
    payments_topic.append(pay)

print(f"Kafka log has {len(payments_topic.log)} events with 30-day retention")
print()

# v1: sums by region
def process_v1(event, output):
    region = event.get("region", "unknown")
    output[region] = output.get(region, 0) + event["amount"]

v1 = KappaProcessor("v1", process_v1)
v1.run(payments_topic)
print()

# v2: new business logic — also count transactions (not just revenue)
# KAPPA: replay same Kafka log from offset 0 with new logic
def process_v2(event, output):
    region = event.get("region", "unknown")
    revenue_key = f"{region}_revenue"
    count_key = f"{region}_count"
    output[revenue_key] = output.get(revenue_key, 0) + event["amount"]
    output[count_key] = output.get(count_key, 0) + 1

print("Deploying v2: replay from offset 0 with new logic")
v2 = KappaProcessor("v2", process_v2)
v2.run(payments_topic, start_offset=0)   # full replay
print()
print("v2 is now caught up — cut over production traffic from v1 to v2")
print("v1 decommissioned. One system to maintain.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Pipeline Observability & SLAs

---

```
SCENARIO:  An on-call DE needs to know: is the pipeline healthy? If not,
           what broke, and where in the pipeline did it fail?

OBSERVABILITY PILLARS:
  Metrics:  count of records processed/failed, throughput (rec/sec), latency
  Logs:     structured logs with run_id, stage_name, record_id on error
  Traces:   end-to-end timing per record through all stages
  Alerts:   PagerDuty/Slack when metric breaches threshold

SLA DEFINITIONS:
  Data freshness SLA:  "daily orders table must be loaded by 06:00 UTC"
  Data quality SLA:   "error rate < 0.1% of records per run"
  Latency SLA:        "streaming p99 latency < 500ms"
  Availability SLA:   "pipeline succeeds on 99.9% of scheduled runs"

ALERT HIERARCHY (noise vs signal):
  CRITICAL (page on-call):   SLA breach imminent, DLQ depth > 1000
  WARNING (Slack):           error rate > 0.05%, consumer lag > 5000
  INFO (dashboard):          run completed, row counts, throughput

DATA FRESHNESS MONITORING:
  SELECT MAX(updated_at) FROM orders  → compare to NOW()
  If gap > 25 hours (missed nightly run) → alert
  This is simpler and more reliable than monitoring the pipeline itself.

DEAD LETTER QUEUE STRATEGY:
  Never silently drop records — always route to DLQ with error context.
  Monitor DLQ depth on a dashboard.
  Set up alert: DLQ receives > 100 records in 5 minutes → investigate.
  DLQ records should be replayable after fixing the root cause.
```

In [ ]:
# PIPELINE OBSERVABILITY SIMULATION

from dataclasses import dataclass, field
from typing import List, Dict, Optional
import time

@dataclass
class SLAConfig:
    freshness_hours: float    # data must be available within N hours
    max_error_rate: float     # fraction of records that may fail (e.g. 0.001)
    max_dlq_depth: int        # alert when dead letter queue exceeds this
    latency_p99_ms: int       # streaming: p99 must be under this

class PipelineMonitor:
    def __init__(self, pipeline_name: str, sla: SLAConfig):
        self.pipeline_name = pipeline_name
        self.sla = sla
        self.dlq: List[Dict] = []
        self.metrics_history: List[Dict] = []

    def record_run(self, run_id: str, processed: int, failed: int,
                   duration_sec: float, last_event_ts: str) -> str:
        error_rate = failed / (processed + failed) if (processed + failed) > 0 else 0
        throughput = processed / duration_sec if duration_sec > 0 else 0

        metric = {
            "run_id": run_id,
            "processed": processed,
            "failed": failed,
            "error_rate": error_rate,
            "throughput_rps": round(throughput, 1),
            "last_event_ts": last_event_ts,
            "dlq_depth": len(self.dlq),
        }
        self.metrics_history.append(metric)

        # SLA checks
        alerts = []
        if error_rate > self.sla.max_error_rate:
            alerts.append(f"CRITICAL: error_rate={error_rate:.3%} > SLA={self.sla.max_error_rate:.3%}")
        if len(self.dlq) > self.sla.max_dlq_depth:
            alerts.append(f"WARNING: DLQ depth={len(self.dlq)} > threshold={self.sla.max_dlq_depth}")

        return alerts

    def send_to_dlq(self, record: Dict, error: str):
        self.dlq.append({"record": record, "error": error, "ts": time.time()})

    def freshness_check(self, last_event_age_hours: float) -> str:
        if last_event_age_hours > self.sla.freshness_hours:
            return f"CRITICAL: data is {last_event_age_hours:.1f}h old, SLA={self.sla.freshness_hours}h"
        return f"OK: data freshness {last_event_age_hours:.1f}h (SLA={self.sla.freshness_hours}h)"

# Demo: orders pipeline health check
sla = SLAConfig(freshness_hours=6, max_error_rate=0.001, max_dlq_depth=100, latency_p99_ms=500)
monitor = PipelineMonitor("orders_etl", sla)

# Simulate some failed records going to DLQ
for i in range(5):
    monitor.send_to_dlq({"order_id": f"bad_{i}"}, "missing required field: amount")

# Record a healthy run
alerts = monitor.record_run("run_20260320", processed=9995, failed=5,
                             duration_sec=120, last_event_ts="2026-03-20T05:30:00")
print("Run metrics:")
m = monitor.metrics_history[-1]
for k, v in m.items():
    print(f"  {k}: {v}")

print(f"\nSLA alerts: {alerts if alerts else ['none — all healthy']}")
print(monitor.freshness_check(last_event_age_hours=2.5))

# Simulate an SLA breach run
print("\n--- Simulating bad run ---")
for i in range(200):
    monitor.send_to_dlq({"id": i}, "schema mismatch")
alerts2 = monitor.record_run("run_20260321", processed=8000, failed=200,
                              duration_sec=300, last_event_ts="2026-03-21T05:00:00")
print(f"SLA alerts: {alerts2}")
print(monitor.freshness_check(last_event_age_hours=8.0))

<a id='10'></a>
## 10. 🗺️ The Data Pipeline Decision Map

```
REQUIREMENT                         ARCHITECTURE           TOOLS
──────────────────────────────────────────────────────────────────────────
Latency > 15 min, simple ops        Batch ETL              Spark, Glue, dbt
Latency 1-15 min                    Micro-batch            Spark Structured Streaming
Latency < 1 min                     Streaming              Flink, Kafka Streams
Need historical + real-time both    Lambda Architecture    Spark + Flink + Druid
One codebase, replay OK             Kappa Architecture     Flink + Kafka (long retention)
Simple triggers, serverless         Event-driven           Lambda + Glue + Step Functions
──────────────────────────────────────────────────────────────────────────

IDEMPOTENCY CHECKLIST:
  □ INSERT OVERWRITE partition (not APPEND)
  □ UPSERT / MERGE with unique key for transactional targets
  □ Advance high-water mark AFTER successful load
  □ DLQ for failed records (never silently drop)
  □ Run deduplication on streaming output (consumer may re-deliver)

INTERVIEW QUESTION: "How would you build a pipeline for X?"
  Step 1: clarify latency SLA (determines batch vs streaming)
  Step 2: clarify data volume (determines tool choice)
  Step 3: clarify consistency requirement (at-least-once vs exactly-once)
  Step 4: propose architecture, draw source→transform→sink flow
  Step 5: address failure modes (idempotency, DLQ, alerting)
```

<a id='11'></a>
## 11. 📋 Interview Cheat Sheet

### When to reach for each pattern:

| Latency Need | Architecture | Key Tech |
|---|---|---|
| > 15 min | Batch ETL | Spark, Glue, dbt |
| 1–15 min | Micro-batch | Spark Structured Streaming |
| < 1 min | True streaming | Flink, Kafka Streams |
| Historical + real-time | Lambda | Batch layer + Speed layer + Serving |
| One system, replay ok | Kappa | Flink + long-retention Kafka |

### The key design decisions:

```
TRIGGER:         cron / event-arrival / file-drop
DELIVERY:        at-most-once / at-least-once / exactly-once
IDEMPOTENCY:     overwrite partition / upsert by key
FAULT RECOVERY:  high-water mark + DLQ + idempotent sink
MONITORING:      data freshness SLA + error rate + DLQ depth
SCALING:         parallelism = Kafka partition count (streaming)
```

### Architecture comparison:

```
                  Batch    Micro-batch   Streaming   Lambda    Kappa
Latency           hours    minutes       seconds     both      seconds
Complexity        low      medium        high        very high high
Historical load   simple   simple        replay      batch     replay
Ops burden        low      medium        high        very high medium
Cost              low      medium        high        high      medium
```

### Gotchas to not forget:

```
❌  APPEND to sink without dedup → duplicate records on retry
✅  INSERT OVERWRITE partition or UPSERT with unique key
❌  Commit Kafka offset BEFORE processing → lose events on crash
✅  Commit AFTER processing (at-least-once); dedup downstream if needed
❌  Advance high-water mark before load succeeds
✅  Advance watermark only on success — safe to re-run if load fails
❌  Lambda architecture: same business logic in both batch and speed layers
✅  Keep logic identical by using same code, or prefer Kappa to avoid duplication
❌  Kafka retention = default 7 days for Kappa
✅  Set Kafka retention = max reprocessing horizon (e.g., 30+ days)
```

<a id='12'></a>
## 12. 🗺️ Summary Map

```
                    DATA PIPELINE DESIGN
                           │
           ┌───────────────┼───────────────┐
           │               │               │
         BATCH         STREAMING         HYBRID
           │               │               │
       scheduled       event-driven    Lambda or
       high throughput  low latency    Kappa arch
       idempotent       stateful       
       Spark/Glue       Flink/Kafka    
           │               │               │
       S3/DW sink      Kafka/Redis    Serving layer
                           │           merges views
                    RELIABILITY
                    ─────────────────────
                    DLQ + idempotency
                    high-water mark
                    SLA monitoring
                    data freshness check
```

---
*End of Data Pipeline Design Master Guide — Sean Edition*